# X-Ray Based Self-Attention Module (XBSAM)

XBSAM combines gradient-guided, periodic-pattern, and parameter-free attention with patch-wise similarity modeling to enhance scattering and diffraction features.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## 1. Gradient Detection Block: Sobel Response

The channel-averaged feature map is processed by fixed horizontal and vertical Sobel kernels to obtain the gradient magnitude.


In [ ]:
self.sobel_x = nn.Conv2d(1,1,3,padding=1,bias=False)
self.sobel_y = nn.Conv2d(1,1,3,padding=1,bias=False)
sobel_x_weight = torch.tensor(
    [[-1, 0, 1],
     [-2, 0, 2],
     [-1, 0, 1]], dtype=torch.float32
).view(1, 1, 3, 3)
sobel_y_weight = torch.tensor(
    [[-1, -2, -1],
     [ 0,  0,  0],
     [ 1,  2,  1]], dtype=torch.float32
).view(1, 1, 3, 3)
self.sobel_x.weight.data = sobel_x_weight
self.sobel_y.weight.data = sobel_y_weight
for param in self.sobel_x.parameters(): param.requires_grad=False
for param in self.sobel_y.parameters(): param.requires_grad=False

gx = self.sobel_x(x_mean)
gy = self.sobel_y(x_mean)
grad = torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)


## 2. Gradient-Guided Feature Modulation

The gradient magnitude is normalized and scaled to form the gradient-guided attention weight.


In [ ]:
g_min, _ = torch.min(grad.view(grad.size(0), grad.size(1), -1), dim=2, keepdim=True)
g_max, _ = torch.max(grad.view(grad.size(0), grad.size(1), -1), dim=2, keepdim=True)
g_min = g_min.view(grad.size(0), grad.size(1), 1, 1)
g_max = g_max.view(grad.size(0), grad.size(1), 1, 1)
grad = (grad - g_min) / (g_max - g_min + 1e-8)
grad = grad * self.gradient_strength + (1.0 - 0.5 * self.gradient_strength)

return x * grad


## 3. Periodic Pattern Detection

Three predefined $3\times3$ kernels are used to characterize different spatial pattern responses. Their parameters are fixed during training.


In [ ]:
self.periodic_conv = nn.Conv2d(1, 3, kernel_size=3, padding=1, bias=False)
kernels = torch.stack([
    torch.tensor([[0, 1, 0],
                  [1,-4, 1],
                  [0, 1, 0]], dtype=torch.float32),
    torch.tensor([[-1,-1,-1],
                  [-1, 8,-1],
                  [-1,-1,-1]], dtype=torch.float32),
    torch.tensor([[1, 0,-1],
                  [0, 0, 0],
                  [-1, 0, 1]], dtype=torch.float32)
], dim=0).view(3, 1, 3, 3)
self.periodic_conv.weight.data = kernels
for param in self.periodic_conv.parameters():
    param.requires_grad = False


## 4. Periodic Attention Modulation

The responses of the predefined kernels are fused by a learnable $1\times1$ convolution followed by a sigmoid operation.


In [ ]:
self.periodic_attention = nn.Sequential(
    nn.Conv2d(3, 1, 1, bias=False),
    nn.Sigmoid()
)

feat = self.periodic_conv(x_mean)
attn = self.periodic_attention(feat)
attn = attn * self.periodic_strength + (1.0 - 0.5 * self.periodic_strength)

return x * attn


## 5. SimAM

SimAM provides parameter-free attention based on the variance energy of feature responses within each channel.


In [ ]:
_, _, h, w = x.size()
n = w * h - 1
x_minus_mu_square = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
y = x_minus_mu_square / (4 * (x_minus_mu_square.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)) + 0.5
return x * self.act(y)


## 6. Feature Enhancement and Patch Extraction

The input features are sequentially enhanced by GDB, PPDB, and SimAM, then downsampled and converted into local patches.


In [ ]:
B, C, H, W = attn_feat.size()
x_mean = attn_feat.mean(dim=1, keepdim=True)
attn_feat = self.gdb(attn_feat, x_mean)
attn_feat = self.ppdb(attn_feat, x_mean)
attn_feat = self.simam(attn_feat)

f_down = F.interpolate(attn_feat, scale_factor=1/self.rate, mode='nearest')
w = self.extract_patches(f_down, self.ksize, self.stride, self.padding)
w = w.view(B, C, self.ksize, self.ksize, -1).permute(0,4,1,2,3)
raw_w = w


## 7. Patch Similarity Response

Normalized local patches are used as convolution kernels to compute similarity responses, followed by softmax weighting.


In [ ]:
Xi = f_down[i:i+1]
wi = w[i]
xi = wi / torch.sqrt((wi**2).sum(dim=(1,2,3), keepdim=True)).clamp(min=1e-4)

Yi = F.conv2d(Xi, xi, padding=self.ksize//2, groups=1)

Yi = F.softmax(Yi * self.softmax_scale, dim=1)


## 8. Feature Mapping

The weighted response is mapped back to the original spatial resolution through transposed convolution and interpolation.


In [ ]:
weight = raw_w[i].permute(0, 1, 2, 3).contiguous()
Yi = F.conv_transpose2d(Yi, weight, stride=self.rate, padding=self.rate//2) / 4.0
Yi = F.interpolate(Yi, size=(H, W), mode='nearest')

Y.append(Yi)

return torch.cat(Y, dim=0)
